# exp_9 — Experiment 9: NLP Sequence Modeling with LSTM and GRU

Source-based notebook converted from the uploaded experiment code document.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout

from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
sample_reviews = [

    "This movie is fantastic",

    "I loved this movie",

    "Worst movie ever",

    "The acting was amazing",

    "The film was boring"

]

tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(sample_reviews)

print("Word Index:\n")

print(tokenizer.word_index)

print("\nTokenized Sequences:\n")

sequences = tokenizer.texts_to_sequences(sample_reviews)

print(sequences)

In [ ]:
padded_sequences = pad_sequences(

    sequences,

    maxlen=6,

    padding="post",

    truncating="post"

)

print("Padded Sequences:\n")

print(padded_sequences)

In [ ]:
vocab_size = 10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

print("Training Samples :", X_train.shape)

print("Testing Samples :", X_test.shape)

In [ ]:
print("First Review (Encoded)\n")

print(X_train[0])

print("\nLength of Review :", len(X_train[0]))

print("\nSentiment Label :", y_train[0])

In [ ]:
word_index = imdb.get_word_index()

reverse_word_index = {
    value + 3: key
    for (key, value) in word_index.items()
}

reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

decoded_review = " ".join(
    reverse_word_index.get(i, "?")
    for i in X_train[0]
)

print(decoded_review)

In [ ]:
max_length = 200

X_train = pad_sequences(

    X_train,

    maxlen=max_length,

    padding="post",

    truncating="post"

)

X_test = pad_sequences(

    X_test,

    maxlen=max_length,

    padding="post",

    truncating="post"

)

print(X_train.shape)

print(X_test.shape)

In [ ]:
print("First Review Shape :", X_train[0].shape)

print("First 30 Tokens:\n")

print(X_train[0][:30])

print("\nSentiment :", y_train[0])

In [ ]:
embedding_dim = 128

lstm_model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_length
    ),

    LSTM(128),

    Dropout(0.5),

    Dense(64, activation="relu"),

    Dense(1, activation="sigmoid")

])

lstm_model.summary()

In [ ]:
lstm_model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
history_lstm = lstm_model.fit(

    X_train,
    y_train,

    validation_data=(X_test, y_test),

    epochs=5,

    batch_size=64,

    verbose=1

)

In [ ]:
loss_lstm, accuracy_lstm = lstm_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("LSTM Test Loss :", loss_lstm)
print("LSTM Test Accuracy :", accuracy_lstm)

In [ ]:
gru_model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_length
    ),

    GRU(128),

    Dropout(0.5),

    Dense(64, activation="relu"),

    Dense(1, activation="sigmoid")

])

gru_model.summary()

In [ ]:
gru_model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history_gru = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)

In [ ]:
loss_gru, accuracy_gru = gru_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("GRU Test Loss :", loss_gru)
print("GRU Test Accuracy :", accuracy_gru)

In [ ]:
loss_gru, accuracy_gru = gru_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("GRU Test Loss :", loss_gru)
print("GRU Test Accuracy :", accuracy_gru)

import pandas as pd

comparison = pd.DataFrame({

    "Model": ["LSTM", "GRU"],

    "Loss": [loss_lstm, loss_gru],

    "Accuracy": [accuracy_lstm, accuracy_gru]

})

print(comparison)

In [ ]:
y_pred_lstm = lstm_model.predict(X_test)

y_pred_lstm = (y_pred_lstm > 0.5).astype(int)

print("LSTM Prediction Shape:", y_pred_lstm.shape)

In [ ]:
y_pred_gru = gru_model.predict(X_test)

y_pred_gru = (y_pred_gru > 0.5).astype(int)

print("GRU Prediction Shape:", y_pred_gru.shape)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm_lstm = confusion_matrix(y_test, y_pred_lstm)

print(cm_lstm)

plt.figure(figsize=(6,6))
plt.imshow(cm_lstm, cmap="Blues")
plt.title("LSTM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()

In [ ]:
cm_gru = confusion_matrix(y_test, y_pred_gru)

print(cm_gru)

plt.figure(figsize=(6,6))
plt.imshow(cm_gru, cmap="Greens")
plt.title("GRU Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print("========== LSTM ==========")

print(classification_report(
    y_test,
    y_pred_lstm,
    target_names=["Negative","Positive"]
))

print("\n========== GRU ==========")

print(classification_report(
    y_test,
    y_pred_gru,
    target_names=["Negative","Positive"]
))

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_lstm.history["accuracy"],
         label="LSTM Train")

plt.plot(history_lstm.history["val_accuracy"],
         label="LSTM Validation")

plt.plot(history_gru.history["accuracy"],
         label="GRU Train")

plt.plot(history_gru.history["val_accuracy"],
         label="GRU Validation")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("LSTM vs GRU Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_lstm.history["loss"],
         label="LSTM Train")

plt.plot(history_lstm.history["val_loss"],
         label="LSTM Validation")

plt.plot(history_gru.history["loss"],
         label="GRU Train")

plt.plot(history_gru.history["val_loss"],
         label="GRU Validation")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM vs GRU Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print("LSTM Training Loss:", len(history_lstm.history["loss"]))
print("LSTM Validation Loss:", len(history_lstm.history["val_loss"]))

print("GRU Training Loss:", len(history_gru.history["loss"]))
print("GRU Validation Loss:", len(history_gru.history["val_loss"]))